# Qwen3-4B-Instruct-2507 LoRA Fine-tuning

**Purpose**: Fine-tune Qwen3-4B-Instruct-2507 on Vietnamese medical QA dataset using LoRA/QLoRA

**Base Model**: Qwen/Qwen3-4B-Instruct-2507

**Dataset**: combined_medical_qa_dataset

**Method**: LoRA (Low-Rank Adaptation) or QLoRA (Quantized LoRA)

**Output**: Fine-tuned model saved to HuggingFace Hub or local directory

## 1. Setup and Imports

In [ ]:
import os
import json
from pathlib import Path

import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_from_disk
import wandb

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

## 2. Configuration

In [ ]:
# Paths
DATA_DIR = Path("../../data/combined_medical_qa")
OUTPUT_DIR = Path("../models/qwen3-4b-medical-lora")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Model configuration
MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"

# LoRA configuration (load from config file)
CONFIG_PATH = Path("../configs/generation_lora_config.yaml")

# Training configuration
USE_QLORA = True  # Set to True for 4-bit quantization (saves VRAM)
MAX_LENGTH = 512
BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4
LEARNING_RATE = 2e-4
NUM_EPOCHS = 3
WARMUP_STEPS = 100
LOGGING_STEPS = 10
SAVE_STEPS = 500

# W&B configuration
WANDB_PROJECT = "vietnamese-medical-rag"
WANDB_RUN_NAME = "qwen3-4b-lora-finetune"

print(f"Model: {MODEL_NAME}")
print(f"Dataset: {DATA_DIR}")
print(f"Output: {OUTPUT_DIR}")
print(f"Use QLoRA: {USE_QLORA}")

## 3. Load Dataset

In [ ]:
# Load dataset
print(f"Loading dataset from {DATA_DIR}...")
dataset = load_from_disk(str(DATA_DIR))

train_dataset = dataset["train"]
val_dataset = dataset["test"]

print(f"\nTrain samples: {len(train_dataset)}")
print(f"Eval samples: {len(val_dataset)}")
print(f"\nSample data:")
print(train_dataset[0])

## 4. Load Tokenizer and Model

In [ ]:
# Load tokenizer
print(f"Loading tokenizer from {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Configure quantization for QLoRA
if USE_QLORA:
    print("Using 4-bit quantization (QLoRA)...")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True
    )
else:
    bnb_config = None

# Load model
print(f"Loading model from {MODEL_NAME}...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config if USE_QLORA else None,
    torch_dtype=torch.float16 if not USE_QLORA else None,
    device_map="auto",
    trust_remote_code=True
)

if USE_QLORA:
    model = prepare_model_for_kbit_training(model)

print(f"\nModel loaded successfully!")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")

## 5. Configure LoRA

In [ ]:
# LoRA configuration
lora_config = LoraConfig(
    r=16,  # LoRA rank
    lora_alpha=32,  # LoRA alpha (scaling factor)
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # Which modules to apply LoRA
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Apply LoRA to model
model = get_peft_model(model, lora_config)

# Print trainable parameters
model.print_trainable_parameters()

print("\nLoRA configuration applied successfully!")

## 6. Prepare Data

In [ ]:
def format_prompt(sample):
    """Format sample as chat messages for Qwen3 instruction tuning.
    
    CRITICAL: Use chat template for Qwen3 models per official guidelines.
    """
    question = sample["question"]
    answer = sample["answer"]
    context = sample.get("context", None)
    
    # Format as chat messages (Qwen3 best practice)
    if context:
        user_message = f"""Dựa vào ngữ cảnh sau, hãy trả lời câu hỏi.

Ngữ cảnh: {context}

Câu hỏi: {question}"""
    else:
        user_message = f"Hãy trả lời câu hỏi sau: {question}"
    
    messages = [
        {"role": "system", "content": "Bạn là trợ lý y tế AI chuyên nghiệp."},
        {"role": "user", "content": user_message},
        {"role": "assistant", "content": answer}
    ]
    
    # Apply chat template
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False  # False because we include the answer
    )
    
    return prompt


def tokenize_function(examples):
    """Tokenize examples with Qwen3 configuration.
    
    Max length: 2048 tokens for training (fits in 24GB VRAM with QLoRA)
    """
    prompts = [format_prompt(sample) for sample in [
        {k: examples[k][i] for k in examples.keys()}
        for i in range(len(examples["question"]))
    ]]
    
    # Tokenize with truncation and padding
    model_inputs = tokenizer(
        prompts,
        max_length=2048,  # From generation_lora_config.yaml
        truncation=True,
        padding="max_length",
        return_tensors=None
    )
    
    # For causal LM, labels are the same as input_ids
    model_inputs["labels"] = model_inputs["input_ids"].copy()
    
    return model_inputs


# Tokenize datasets
print("Tokenizing datasets with Qwen3 chat template...")
tokenized_train = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=train_dataset.column_names,
    desc="Tokenizing training data"
)

tokenized_val = val_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=val_dataset.column_names,
    desc="Tokenizing validation data"
)

print(f"Tokenized training samples: {len(tokenized_train)}")
print(f"Tokenized validation samples: {len(tokenized_val)}")
print(f"\nSample tokenized input length: {len(tokenized_train[0]['input_ids'])} tokens")

## 7. Training Configuration

In [ ]:
# Training arguments
training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    eval_strategy="steps",
    eval_steps=SAVE_STEPS,
    save_total_limit=3,
    fp16=True,
    optim="paged_adamw_8bit" if USE_QLORA else "adamw_torch",
    report_to="wandb",
    run_name=WANDB_RUN_NAME,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False
)

# Initialize W&B
wandb.init(
    project=WANDB_PROJECT,
    name=WANDB_RUN_NAME,
    config={
        "model": MODEL_NAME,
        "lora_r": lora_config.r,
        "lora_alpha": lora_config.lora_alpha,
        "use_qlora": USE_QLORA,
        "max_length": MAX_LENGTH,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "num_epochs": NUM_EPOCHS
    }
)

print("Training configuration ready!")

## 8. Train Model

In [ ]:
# Create trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    val_dataset=val_dataset
)

# Train
print("Starting training...")
trainer.train()

print("\n✓ Training complete!")

## 9. Save Model

In [ ]:
# Save final model
trainer.model.save_pretrained(str(OUTPUT_DIR / "finetuned_model"))
tokenizer.save_pretrained(str(OUTPUT_DIR / "finetuned_model"))

print(f"\nModel saved to: {OUTPUT_DIR / 'finetuned_model'}")

# Finish W&B run
wandb.finish()

print("\n✓ Fine-tuning complete!")
print(f"\nNext steps:")
print(f"1. Run evaluation notebook (05_evaluation.ipynb) to compare with baseline")
print(f"2. Upload model to HuggingFace Hub using ml/scripts/upload_to_hub.py")
print(f"3. Update backend/src/services/brain.py to use fine-tuned model")